# 🔭 Détecteurs & Pixels touchés — Fink/LSST

Ce notebook interroge l'endpoint **`/api/v1/sources`** de Fink/LSST pour une plage de dates définie et produit :

1. **Histogramme des détecteurs touchés** (`detector`) — combien d'alertes ont été enregistrées sur chaque CCD.
2. **Carte 2D des pixels touchés** (`x`, `y`) — densité des positions de centroïdes sur le plan focal.

**API :** `https://api.lsst.fink-portal.org`  
**Endpoint utilisé :** `/api/v1/sources`  
**Colonnes clés :** `r:detector`, `r:x`, `r:y`, `r:midpointMjdTai`, `r:band`

## 1. Paramètres

In [ ]:
# ─── Centre du champ ──────────────────────────────────────────────────────────
# Option A : coordonnées décimales
TARGET_RA   = 53.16    # degrés décimaux
TARGET_DEC  = -28.10  # degrés décimaux
TARGET_NAME = None        # ex. "NGC 1365" — si renseigné, résout le nom et ignore RA/Dec

# ─── Rayon de recherche ───────────────────────────────────────────────────────
RADIUS_ARCSEC = 1*3600.0     # arcsec (max 18000 = 5 deg)

# ─── Bornes temporelles (optionnel) ───────────────────────────────────────────
STARTDATE  = None         # UTC/ISO "2025-10-01 00:00:00" ou None
ENDDATE   = None         # UTC/ISO "2026-03-15 00:00:00" ou None
# Alternative : WINDOW = 30  # jours depuis STARTDATE

# Minimum number of alerts (diaSources) per sky objects (diaObjectsId)
MIN_NUMBER_SOURCES = 1  # intger or None (no cut)

# Maximum sky objects to take into accounts
MAX_OBJECTS = 100

# ─── Filtrage optionnel par bande ─────────────────────────────────────────────
BANDS_FILTER = None   # ex. ['r', 'i'] ou None pour toutes les bandes

# ─── Options visuelles ────────────────────────────────────────────────────────
HIST_BINS      = None   # None = auto (un bin par détecteur)
PIXEL_BINS     = 200    # résolution de la carte 2D (200×200)
PIXEL_CMAP     = 'inferno'
DETECTOR_CMAP  = 'viridis'

# ─── Sauvegarde ───────────────────────────────────────────────────────────────
SAVE_FIGS  = True
OUTPUT_DIR = "detector_outputs"

BASE_URL = "https://api.lsst.fink-portal.org"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from matplotlib.ticker import MaxNLocator

from util_date import *
from util_lc_cutout_plot import FILTER_COLORS


warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 12,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'legend.fontsize'   : 10,
    'figure.titlesize'  : 14,
    'figure.titleweight': 'bold',
})

MJDTAI_START = datestr_iso_utc_to_mjd_tai(STARTDATE)
MJDTAI_END   = datestr_iso_utc_to_mjd_tai(ENDDATE)

print("invalidation de STARTDATE/ENDDATE pour creer error, USE MJDTAI_START/MJDTAI_END")
del STARTDATE
del ENDDATE


print("Imports OK ✓")

# CCD geometry Focal Plane

In [ ]:
geometry_df = pd.read_csv("ccd_geometry.csv")

## 3. Récupération des sources via `/api/v1/conesearch`

In [ ]:
COLUMNS = [
    'r:detector',
    'r:visit',
    'r:x',
    'r:y',
    'r:midpointMjdTai',
    'r:band',
    'r:diaObjectId',
    'r:diaSourceId',
    'r:ra',
    'r:dec',
    'r:snr',
    'r:band',
    'r:nDiaSources',
    'r:psfFlux',
]
tags = [c.replace('r:', '') for c in COLUMNS]

payload = {
    "n"     : MAX_OBJECTS,
    "ra"    : str(TARGET_RA),
    "dec"   : str(TARGET_DEC),
    "radius": str(RADIUS_ARCSEC),
    "columns": ",".join(COLUMNS),
    "output-format": "json",
}
if MJDTAI_START: payload["startdate"] = MJDTAI_START
if MJDTAI_END:  payload["stopdate"]  = MJDTAI_END

resp = requests.post(f"{BASE_URL}/api/v1/conesearch", json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()

if not data:
    print("⚠️  Aucun objet trouvé dans ce champ.")
    df  = pd.DataFrame()
    object_ids = []
else:
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '').replace('v:', '') for c in df.columns]

SKIP_COLS = {'band', 'diaObjectId', 'diaSourceId', 'nDiaSources','visit'}
for col in df.columns:
    if col in SKIP_COLS:
        continue
    try:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    except Exception:
        pass



## 4. Filtrage et résumé

In [ ]:
df_clean = df.dropna(subset=['detector', 'x', 'y']).copy()

# Filtrage optionnel par bande
if BANDS_FILTER and 'band' in df_clean.columns:
    df_clean = df_clean[df_clean['band'].isin(BANDS_FILTER)]
    print(f"Filtré sur bandes : {BANDS_FILTER}")

# Plage MJD réelle
if 'midpointMjdTai' in df_clean.columns:
    mjd_min = df_clean['midpointMjdTai'].min()
    mjd_max = df_clean['midpointMjdTai'].max()
    date_min = mjd_tai_to_datestr_iso_utc(mjd_min)
    date_max = mjd_tai_to_datestr_iso_utc(mjd_max)
else:
    date_min, date_max = MJDTAI_START, MJDTAI_END

n_sources  = len(df_clean)
n_det      = df_clean['detector'].nunique() if 'detector' in df_clean.columns else '?'
n_bands    = df_clean['band'].nunique()     if 'band'     in df_clean.columns else '?'

print(f"""
╔══════════════════════════════════════════════╗
  Sources valides  : {n_sources:>8}
  Période couverte : {date_min}  →  {date_max}
  Détecteurs actifs: {n_det}
  Bandes           : {n_bands}
╚══════════════════════════════════════════════╝
""")

if 'band' in df_clean.columns:
    print("Répartition par bande :")
    print(df_clean['band'].value_counts().to_string())

In [ ]:
df_clean.keys()

In [ ]:
df_tmp = df_clean[df_clean['detector']==94].head(10)

In [ ]:
df_tmp.keys()

In [ ]:
df_tmp.to_csv("alerts_dect94.csv", index=False)

## 5. Histogramme des détecteurs touchés

In [ ]:
det_counts = df_clean['detector'].value_counts().sort_index()
det_ids    = det_counts.index.astype(int)
det_vals   = det_counts.values

# Couleur proportionnelle au nombre d'alertes
cmap_det = plt.get_cmap(DETECTOR_CMAP)
norm_det = mcolors.Normalize(vmin=det_vals.min(), vmax=det_vals.max())
colors   = [cmap_det(norm_det(v)) for v in det_vals]

fig, ax = plt.subplots(figsize=(max(14, len(det_ids) * 0.3), 5), layout='constrained')

bars = ax.bar(det_ids, det_vals, color=colors, edgecolor='white', linewidth=0.4, width=0.85)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_det, norm=norm_det)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Nombre d\'alertes', shrink=0.6, pad=0.01)

ax.set_xlabel('ID Détecteur (CCD)')
ax.set_ylabel('Nombre d\'alertes')
ax.set_title(
    f'Histogramme des détecteurs touchés\n'
    f'{date_min} → {date_max}  |  {n_sources:,} sources  |  {n_det} détecteurs actifs'
)
ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=20))
ax.grid(True, axis='y', alpha=0.3, linewidth=0.5)
ax.set_xlim(det_ids.min() - 1, det_ids.max() + 1)

# Annotation du détecteur le plus actif
peak_det = det_counts.idxmax()
peak_val = det_counts.max()
ax.annotate(
    f'max : det {peak_det}\n{peak_val} alertes',
    xy=(peak_det, peak_val),
    xytext=(12, 8), textcoords='offset points',
    fontsize=9, color='white',
    bbox=dict(boxstyle='round,pad=0.3', fc='#333333', alpha=0.8),
    arrowprops=dict(arrowstyle='->', color='white', lw=0.9)
)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/detector_histogram.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/detector_histogram.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/detector_histogram.pdf / .png")

plt.show()

## 6. Histogramme des détecteurs — décomposé par bande

In [ ]:
if 'band' in df_clean.columns:
    bands_present = sorted(df_clean['band'].dropna().unique())
    all_det = sorted(df_clean['detector'].dropna().astype(int).unique())

    fig, ax = plt.subplots(figsize=(max(14, len(all_det) * 0.3), 5), layout='constrained')

    bottom = np.zeros(len(all_det))
    det_index = {d: i for i, d in enumerate(all_det)}

    for band in bands_present:
        sub = df_clean[df_clean['band'] == band]
        cnt_band = sub['detector'].astype(int).value_counts()
        vals = np.array([cnt_band.get(d, 0) for d in all_det])
        color = FILTER_COLORS.get(band, '#888888')
        ax.bar(all_det, vals, bottom=bottom, color=color, alpha=0.85,
               edgecolor='white', linewidth=0.3, width=0.85, label=f'Bande {band}')
        bottom += vals

    ax.set_xlabel('ID Détecteur (CCD)')
    ax.set_ylabel('Nombre d\'alertes')
    ax.set_title(
        f'Détecteurs touchés — décomposition par bande\n'
        f'{date_min} → {date_max}  |  {n_sources:,} sources'
    )
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=20))
    ax.grid(True, axis='y', alpha=0.3, linewidth=0.5)
    ax.set_xlim(min(all_det) - 1, max(all_det) + 1)
    ax.legend(loc='upper right', ncol=len(bands_present))

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/detector_histogram_by_band.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/detector_histogram_by_band.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé : {OUTPUT_DIR}/detector_histogram_by_band.pdf / .png")

    plt.show()
else:
    print("Colonne 'band' absente — graphique par bande non disponible.")

# Focal Plane Heat Map

In [ ]:
def plot_focal_plane_heatmap(
    alerts_df,
    geom, # geometr_cdd df
    value_col=None,  # None = counts, sinon colonne à agréger
    agg_func="mean",  # "count", "mean", etc.
    cmap="viridis",
    log_scale=False,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
):
    """
    Affiche une heatmap de la surface focale LSST.

    Parameters
    ----------
    alerts_df : pd.DataFrame
        Doit contenir une colonne 'detector'
    geom : ccd_geometry.csv as pd.dataframe
    value_col : str or None
        Si None → compte le nombre d’objets par CCD
        Sinon → agrège cette colonne
    agg_func : str
        Fonction d’agrégation ('count', 'mean', etc.)
    """

    # --- Load geometry
    #geom = pd.read_csv(geometry_csv)
    
    # --- Compute values per detector
    if value_col is None:
        values = alerts_df.groupby("detector").size()
    else:
        values = alerts_df.groupby("detector")[value_col].agg(agg_func)

    # --- Merge with geometry
    geom["value"] = geom["detector"].map(values).fillna(0)

    # --- Optional log scale
    if log_scale:
        geom["value"] = np.log10(geom["value"] + 1)

    # --- Normalize colormap
    vmin = geom["value"].min()
    vmax = geom["value"].max()
    
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap)

    # --- Plot
    fig, ax = plt.subplots(figsize=figsize)

    
    for _, row in geom.iterrows():
        corners = [
            (row["corner0_x"], row["corner0_y"]),
            (row["corner1_x"], row["corner1_y"]),
            (row["corner2_x"], row["corner2_y"]),
            (row["corner3_x"], row["corner3_y"]),
        ]

        color = cmap(norm(row["value"]))

        poly = patches.Polygon(corners, facecolor=color, edgecolor="black", linewidth=0.2)

        ax.add_patch(poly)
        #ax.scatter(row["x_center"],row["y_center"],s=50,c=color)

    # --- Aesthetic
    ax.set_aspect("equal")
    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")
    ax.set_title(title)

    # Remove axes ticks (optional)
    #ax.set_xticks([])
    #ax.set_yticks([])

    ax.set_xlim([-400,400])
    ax.set_ylim([-400,400])
    
    # --- Colorbar
    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/focal_plane_hit_map.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/focal_plane_hit_map.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé : {OUTPUT_DIR}/focal_plane_hit_map.pdf / .png")


In [ ]:
plot_focal_plane_heatmap(
    df_clean,
    geometry_df, # geometr_cdd df
    value_col=None,  # None = counts, sinon colonne à agréger
    agg_func="count",  # "count", "mean", etc.
    cmap="viridis",
    log_scale=False,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
)

## 7. Carte 2D des pixels touchés (x, y)

In [ ]:
x = df_clean['x'].dropna().values
y = df_clean['y'].dropna().values

# Filtre cohérent (même index)
mask = df_clean['x'].notna() & df_clean['y'].notna()
x = df_clean.loc[mask, 'x'].values
y = df_clean.loc[mask, 'y'].values

print(f"Points valides pour la carte (x,y) : {len(x):,}")
print(f"  x : [{x.min():.1f}, {x.max():.1f}]  pixel")
print(f"  y : [{y.min():.1f}, {y.max():.1f}]  pixel")

# Histogramme 2D
H, xedges, yedges = np.histogram2d(x, y, bins=PIXEL_BINS)
H = H.T   # transposition pour que x→colonnes, y→lignes (convention imshow)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
fig.suptitle(
    f'Carte des pixels touchés (x, y)\n'
    f'{date_min} → {date_max}  |  {len(x):,} sources'
)

# ── Échelle linéaire ──────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(
    H, origin='lower', aspect='equal',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap=PIXEL_CMAP, interpolation='nearest'
)
fig.colorbar(im, ax=ax, label='Nombre d\'alertes (linéaire)', shrink=0.8)
ax.set_xlabel('x  [pixel]')
ax.set_ylabel('y  [pixel]')
ax.set_title('Échelle linéaire')
ax.grid(False)

# ── Échelle log ───────────────────────────────────────────────────────────────
ax = axes[1]
H_log = np.where(H > 0, H, np.nan)
im2 = ax.imshow(
    H_log, origin='lower', aspect='equal',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap=PIXEL_CMAP,
    norm=mcolors.LogNorm(vmin=1, vmax=H.max()),
    interpolation='nearest'
)
fig.colorbar(im2, ax=ax, label='Nombre d\'alertes (log)', shrink=0.8)
ax.set_xlabel('x  [pixel]')
ax.set_ylabel('y  [pixel]')
ax.set_title('Échelle logarithmique')
ax.grid(False)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/pixel_map_xy.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/pixel_map_xy.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/pixel_map_xy.pdf / .png")

plt.show()

## 8. Carte scatter colorée par bande (sous-échantillonnée)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8), layout='constrained')

MAX_SCATTER = 10000   # limite de points pour la lisibilité

if 'band' in df_clean.columns:
    for band in sorted(df_clean['band'].dropna().unique()):
        sub = df_clean[df_clean['band'] == band].dropna(subset=['x','y'])
        if len(sub) > MAX_SCATTER:
            sub = sub.sample(MAX_SCATTER, random_state=42)
        color = FILTER_COLORS.get(band, '#888888')
        ax.scatter(
            sub['x'], sub['y'],
            s=1.5, alpha=0.35, color=color, label=f'Bande {band} ({len(sub):,})'
        )
    ax.legend(markerscale=5, loc='upper right')
else:
    sub = df_clean.dropna(subset=['x','y'])
    if len(sub) > MAX_SCATTER:
        sub = sub.sample(MAX_SCATTER, random_state=42)
    ax.scatter(sub['x'], sub['y'], s=1, alpha=0.3, color='steelblue')

ax.set_xlabel('x  [pixel]')
ax.set_ylabel('y  [pixel]')
ax.set_title(
    f'Positions (x, y) des alertes — par bande\n'
    f'{date_min} → {date_max}  (≤ {MAX_SCATTER} pts/bande)'
)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2, linewidth=0.5)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/pixel_scatter_by_band.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/pixel_scatter_by_band.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/pixel_scatter_by_band.pdf / .png")

plt.show()

## 9. Top-10 détecteurs les plus actifs

In [ ]:
top10 = det_counts.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')

cmap_top = plt.get_cmap('plasma')
norm_top = mcolors.Normalize(vmin=top10.values.min(), vmax=top10.values.max())
colors_top = [cmap_top(norm_top(v)) for v in top10.values]

ax.barh(
    [f"Det {int(d)}" for d in top10.index],
    top10.values,
    color=colors_top, edgecolor='white', linewidth=0.4
)
ax.invert_yaxis()
ax.set_xlabel('Nombre d\'alertes')
ax.set_title(
    f'Top-10 détecteurs les plus actifs\n'
    f'{date_min} → {date_max}'
)

# Annoter les valeurs
for i, (det, val) in enumerate(zip(top10.index, top10.values)):
    ax.text(val + top10.values.max() * 0.01, i, f'{val:,}',
            va='center', fontsize=9)

ax.grid(True, axis='x', alpha=0.3)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/top10_detectors.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/top10_detectors.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/top10_detectors.pdf / .png")

plt.show()

print("\nTop-10 détecteurs :")
print(top10.to_string())

## 10. Résumé tabulaire

In [ ]:
from IPython.display import display as ipy_display, HTML

summary = df_clean.groupby('detector').agg(
    n_alertes  = ('diaSourceId', 'count') if 'diaSourceId' in df_clean.columns else ('x', 'count'),
    x_mean     = ('x', 'mean'),
    x_std      = ('x', 'std'),
    y_mean     = ('y', 'mean'),
    y_std      = ('y', 'std'),
    snr_median = ('snr', 'median') if 'snr' in df_clean.columns else ('x', lambda _: np.nan),
).reset_index()

summary['detector'] = summary['detector'].astype(int)
summary = summary.sort_values('n_alertes', ascending=False)

print(f"Résumé par détecteur ({len(summary)} CCDs actifs) :")
ipy_display(summary.style
    .format({'x_mean': '{:.1f}', 'x_std': '{:.1f}',
             'y_mean': '{:.1f}', 'y_std': '{:.1f}',
             'snr_median': '{:.2f}'})
    .background_gradient(subset=['n_alertes'], cmap='YlOrRd')
)

#if SAVE_FIGS:
#    summary.to_csv(f"{OUTPUT_DIR}/detector_summary.csv", index=False)
#    print(f"\n✓ CSV sauvegardé : {OUTPUT_DIR}/detector_summary.csv")